# Lab 05｜問卷能代表全校嗎 Sampling Bias

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-05-sampling-bias.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：學校沒有時間詢問所有人。哪一種抽樣方式較可能代表合成母體？

- 比較 **simple random sample (SRS)**、**stratified random sample** 與 convenience/voluntary response sample。
- 用 sample statistic − population parameter 定義本 Lab 的 **sampling error**。
- 說明 sampling variability 與 sampling bias 不相同。


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_routine_survey.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 把完整合成資料當作已知母體

這只是模擬環境；真實調查通常不知道 population mean。`SAMPLE_SIZE` 請先保持為 4 的倍數，方便按四個年級等額分層。


In [ ]:
# ✏️ 修改任務：完成 n=40 後，改成 20 或 80，比較 sampling variability。
OUTCOME = "readiness_score"
SAMPLE_SIZE = 40
SAMPLING_SEED = 115

if SAMPLE_SIZE <= 0 or SAMPLE_SIZE > len(data) or SAMPLE_SIZE % 4 != 0:
    raise ValueError("SAMPLE_SIZE 必須是 4 的正倍數，且不可超過母體筆數。")

population_mean = data[OUTCOME].mean()
print(f"Population size N = {len(data)}")
print(f"Population mean readiness = {population_mean:.2f} points")


### 2. 建立三種樣本

`newsletter_link` 是刻意設計的自願回覆管道，其參與傾向與準備度有關，因此可示範 selection bias。


In [ ]:
srs = data.sample(n=SAMPLE_SIZE, random_state=SAMPLING_SEED)

per_grade = SAMPLE_SIZE // data["grade"].nunique()
stratified = (
    data.groupby("grade", group_keys=False)
    .sample(n=per_grade, random_state=SAMPLING_SEED)
)

volunteer_pool = data.query("response_source == 'newsletter_link'")
convenience = volunteer_pool.sample(n=SAMPLE_SIZE, random_state=SAMPLING_SEED)

samples = {
    "SRS": srs,
    "Stratified by grade": stratified,
    "Newsletter volunteers": convenience,
}

sample_summary = pd.DataFrame([
    {
        "method": method,
        "n": len(sample),
        "sample_mean": sample[OUTCOME].mean(),
        "error_sample_minus_population": sample[OUTCOME].mean() - population_mean,
    }
    for method, sample in samples.items()
])
display(sample_summary.round(2))


### 3. 重複抽取許多 SRS

單一 SRS 可能偏高或偏低；重複抽樣後，樣本平均數會在母體平均數兩側變動。


In [ ]:
SIMULATIONS = 500
rng = np.random.default_rng(SAMPLING_SEED)
population_values = data[OUTCOME].dropna().to_numpy()
srs_means = np.array([
    rng.choice(population_values, size=SAMPLE_SIZE, replace=False).mean()
    for _ in range(SIMULATIONS)
])

FIGURE_ALT = (
    f"Histogram of {SIMULATIONS} SRS mean readiness scores for samples of size "
    f"{SAMPLE_SIZE}, centered near the synthetic population mean."
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(srs_means, bins=22, color="#6A4C93", edgecolor="white", ax=ax)
ax.axvline(population_mean, color="#D62828", linewidth=2, label="Population mean")
ax.set_title(f"Sampling variability of SRS means (n={SAMPLE_SIZE})")
ax.set_xlabel("Sample mean learning readiness（points）")
ax.set_ylabel("Number of simulated SRS samples（次數）")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Alt text: {FIGURE_ALT}")
print(f"SD of simulated sample means = {srs_means.std(ddof=1):.2f} points")


<details>
<summary><strong>AP English Response frame</strong></summary>

> An SRS of ____ students gives each possible sample of that size an equal chance of selection. The stratified design guarantees ____ students from each grade. The newsletter sample may be biased because ____. Therefore, results from ____ are more defensible for generalizing to this synthetic population.

</details>

注意：較大的便利樣本仍可能有偏誤；增加樣本數不能修復選取機制的系統性問題。


## Checks

確認三種樣本大小、分層配置與模擬次數。


In [ ]:
assert all(len(sample) == SAMPLE_SIZE for sample in samples.values())
assert set(stratified["grade"].value_counts()) == {per_grade}
assert len(srs_means) == SIMULATIONS
assert np.isfinite(srs_means).all()
print("✅ Checks passed：三種樣本大小一致，分層樣本各年級人數相等。")


## Next Steps

抽樣處理「可以推廣給誰」；實驗設計處理「可以作因果結論嗎」。Lab 06 將分析年級區集內的隨機分派晨間規劃試驗。
